# Lead / Backing Vocal Separation — 메인보컬 vs 화음/백보컬 분리

한 곡을 입력받아 **메인 보컬과 백보컬(화음)을 분리**합니다. 2단계 파이프라인:

| 단계 | 모델 | 출력 |
|---|---|---|
| 1️⃣ | **Demucs htdemucs** | vocals (메인+화음 합쳐진 사람 목소리 전체) / instrumental |
| 2️⃣ | **UVR-MDX-NET-Karaoke** | vocals를 → lead vocal / backing vocal 로 다시 분리 |

두 단계로 나누는 이유: htdemucs 단독으론 보컬 안에서 lead/backing을 구분 못 함. UVR Karaoke 모델은 보컬 트랙에서 lead/backing을 학습한 모델이라 한 번 더 통과시킴.

**환경**: Colab GPU 권장 (T4 무료티어로 충분). 곡 1개당 30~60초 정도.

## 사용법
1. 런타임 → 런타임 유형 변경 → **GPU** 선택 후 셀 순서대로 실행
2. **§3**에서 음원 업로드
3. **§4**에서 분리 실행
4. **§5**에서 재생 / **§6**에서 ZIP 다운로드

## 0. 의존성 설치

처음 한 번만 실행. `audio-separator`가 UVR 모델들을 ONNX/PyTorch로 래핑해주는 패키지.

In [ ]:
%pip install -q "audio-separator[gpu]" demucs soundfile numpy ipython

## 1. GPU 확인

Colab 런타임이 GPU에 연결됐는지 점검. `False`면 메뉴 → 런타임 → 런타임 유형 변경에서 GPU 선택 후 재시작.

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. 출력 폴더 준비

In [ ]:
from pathlib import Path

WORK_DIR = Path("/content/separation")
INPUT_DIR = WORK_DIR / "input"
STAGE1_DIR = WORK_DIR / "stage1_demucs"   # vocals / instrumental
STAGE2_DIR = WORK_DIR / "stage2_uvr"      # lead / backing
FINAL_DIR = WORK_DIR / "final"            # 정리된 최종 결과

for d in [INPUT_DIR, STAGE1_DIR, STAGE2_DIR, FINAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_DIR:", WORK_DIR)

## 3. 음원 로드

Colab 좌측 폴더 사이드바에서 `/content/`에 분석할 음원을 드래그 앤 드롭으로 올린 뒤, 아래 `LOCAL_AUDIO_PATH`를 그 경로로 지정해 셀을 실행하세요.

예: `/content/내곡.mp3`

In [ ]:
from pathlib import Path
import shutil

# 사이드바에서 /content/에 올린 파일 경로를 지정
LOCAL_AUDIO_PATH = '/content/곡.mp3'

src = Path(LOCAL_AUDIO_PATH)
if not src.exists():
    raise FileNotFoundError(
        f'{LOCAL_AUDIO_PATH} 없음 — Colab 좌측 폴더에 음원을 올린 뒤 경로 맞추세요.'
    )

# INPUT_DIR로 복사 — 이후 단계가 input_path를 사용
input_path = INPUT_DIR / src.name
if src.resolve() != input_path.resolve():
    shutil.copy(src, input_path)
STEM = input_path.stem
print(f'입력: {input_path}')
print(f'Stem: {STEM}')
print(f'크기: {input_path.stat().st_size / 1e6:.1f} MB')

## 4. 분리 실행

### 4.1 Stage 1 — Demucs로 보컬과 반주 분리

`htdemucs` 모델로 vocals / drums / bass / other 4-stem 분리. 우리는 **vocals만** 다음 단계로 넘기고, drums+bass+other는 합쳐 instrumental 트랙으로 보관.

In [ ]:
import subprocess

# demucs CLI 사용 — GPU 자동 감지
cmd = [
    "demucs",
    "-n", "htdemucs",
    "--two-stems", "vocals",   # vocals vs no_vocals(반주) 두 트랙으로
    "-o", str(STAGE1_DIR),
    str(input_path),
]
print("실행:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-500:] if result.stdout else "")
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
    raise RuntimeError("Demucs 분리 실패")

# demucs는 stage1/htdemucs/<stem>/ 아래에 vocals.wav, no_vocals.wav를 만든다
demucs_out = STAGE1_DIR / "htdemucs" / STEM
vocals_path = demucs_out / "vocals.wav"
instrumental_path = demucs_out / "no_vocals.wav"
assert vocals_path.exists(), f"vocals.wav 없음: {vocals_path}"
assert instrumental_path.exists(), f"no_vocals.wav 없음: {instrumental_path}"
print(f"vocals: {vocals_path} ({vocals_path.stat().st_size / 1e6:.1f} MB)")
print(f"instrumental: {instrumental_path} ({instrumental_path.stat().st_size / 1e6:.1f} MB)")

### 4.2 Stage 2 — UVR Karaoke 모델로 lead vs backing 분리

`UVR-MDX-NET-Karaoke` 모델은 보컬 트랙에서 메인 보컬(가운데 정위)과 백보컬·화음(좌우로 펼쳐진 것)을 분리하도록 학습됐어요. 모델 가중치(~150MB)는 첫 실행 시 자동 다운로드.

`audio-separator`가 알아서 GPU(`onnxruntime-gpu`)를 잡습니다.

In [ ]:
from audio_separator.separator import Separator

# Karaoke 모델 — 메인보컬 vs 백보컬·화음
KARAOKE_MODEL = "UVR-MDX-NET-Karaoke.onnx"

separator = Separator(
    output_dir=str(STAGE2_DIR),
    output_format="WAV",
    normalization_threshold=0.9,
)
separator.load_model(model_filename=KARAOKE_MODEL)

# vocals.wav를 입력으로 — 출력은 (instrumental_part, vocals_part) 두 개
# Karaoke 모델 컨벤션: "_(Vocals)_" 가 lead, "_(Instrumental)_" 가 backing(화음)
uvr_outputs = separator.separate(str(vocals_path))
print("UVR 출력 파일들:")
for p in uvr_outputs:
    full = STAGE2_DIR / p if not Path(p).is_absolute() else Path(p)
    print("  -", full)

### 4.3 결과 정리

UVR 모델의 출력 파일명 컨벤션이 좀 어지러워서(`<stem>_(Vocals)_<model>.wav` 식) 의미가 명확한 이름으로 복사:

- `<stem>_lead.wav` — 메인 보컬
- `<stem>_backing.wav` — 백보컬 / 화음
- `<stem>_vocals.wav` — Demucs 단계의 보컬 전체 (참고용)
- `<stem>_instrumental.wav` — Demucs 단계의 반주

Karaoke 모델은 "Vocals" 라벨을 lead에, "Instrumental" 라벨을 backing에 붙입니다 (모델 입장에서 backing은 메인의 "반주"처럼 보이기 때문).

In [ ]:
import shutil

# UVR 출력 파일 두 개 식별
uvr_files = sorted(STAGE2_DIR.glob("*.wav"))
lead_src = next((p for p in uvr_files if "(Vocals)" in p.name), None)
backing_src = next((p for p in uvr_files if "(Instrumental)" in p.name), None)

if lead_src is None or backing_src is None:
    print("⚠️ 예상한 파일명을 못 찾음. UVR 출력:")
    for p in uvr_files:
        print("  -", p.name)
    raise RuntimeError("파일명 패턴 불일치 — 코드 수정 필요")

lead_path = FINAL_DIR / f"{STEM}_lead.wav"
backing_path = FINAL_DIR / f"{STEM}_backing.wav"
vocals_final = FINAL_DIR / f"{STEM}_vocals.wav"
inst_final = FINAL_DIR / f"{STEM}_instrumental.wav"

shutil.copy(lead_src, lead_path)
shutil.copy(backing_src, backing_path)
shutil.copy(vocals_path, vocals_final)
shutil.copy(instrumental_path, inst_final)

for p in [lead_path, backing_path, vocals_final, inst_final]:
    print(f"✅ {p.name} ({p.stat().st_size / 1e6:.1f} MB)")

## 4.5 (대안) Audioshake API로 lead/backing 재분리

Audioshake는 fetch 방식 API라 vocals.wav가 외부 다운로드 가능한 URL이어야 합니다. 외부 임시 호스트(0x0.st 등)는 31MB 업로드가 자주 끊기니, **Drive의 공개 폴더**에 올린 뒤 Drive direct URL로 audioshake에 넘기는 흐름으로 갑니다.

**1회 수동 설정 (한 번만)**:
1. Drive 웹(`drive.google.com`)에서 `audioshake_uploads/` 폴더가 생긴 뒤 (첫 셀 실행 후)
2. 그 폴더 우클릭 → **공유** → "**링크가 있는 모든 사용자**" (뷰어 권한)로 변경

이후엔 그 폴더에 올린 파일은 자동으로 공개돼서 audioshake가 fetch 가능. 1회만 하면 됨.

흐름:
- **입력**: Demucs vocals.wav (§4.1)
- vocals.wav를 Drive `/audioshake_uploads/`에 복사 → FILE_ID 조회 → direct URL 구성
- audioshake `POST /tasks` 호출 → lead/backing wav 다운로드
- 결과는 `FINAL_DIR`에 저장 (§5/§6에서 자동 표시)

In [ ]:
# Audioshake API 키 로드
# 우선순위: Colab Secrets → 환경변수 → 직접 입력 프롬프트
import os

AUDIOSHAKE_API_KEY = None

# 1) Colab Secrets (왼쪽 사이드바 🔑 아이콘 → 'AUDIOSHAKE_API_KEY' 추가)
try:
    from google.colab import userdata
    AUDIOSHAKE_API_KEY = userdata.get('AUDIOSHAKE_API_KEY')
except Exception:
    pass

# 2) 환경변수
if not AUDIOSHAKE_API_KEY:
    AUDIOSHAKE_API_KEY = os.environ.get('AUDIOSHAKE_API_KEY')

# 3) 그래도 없으면 직접 입력 (마스킹 안 됨에 주의)
if not AUDIOSHAKE_API_KEY:
    from getpass import getpass
    AUDIOSHAKE_API_KEY = getpass('Audioshake API key: ').strip()

if not AUDIOSHAKE_API_KEY:
    raise RuntimeError('API 키가 필요합니다.')

# base URL — 본인 계정 dashboard에서 확인 후 필요시 변경
AUDIOSHAKE_BASE = 'https://groovy.audioshake.ai'

print(f'API 키 로드됨 (length={len(AUDIOSHAKE_API_KEY)})')
print(f'Base URL: {AUDIOSHAKE_BASE}')

In [ ]:
import requests, time, shutil
from pathlib import Path
from google.colab import drive, auth
from googleapiclient.discovery import build

assert vocals_path.exists(), f'vocals.wav가 없음: {vocals_path} — §4.1 먼저 실행하세요.'

# 1) Drive 마운트 + Drive API 인증
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
auth.authenticate_user()  # Drive API용 OAuth
drive_svc = build('drive', 'v3')

# 2) 공개 폴더 확보 — 없으면 생성, 있으면 ID 조회
FOLDER_NAME = 'audioshake_uploads'
q = (
    f"name='{FOLDER_NAME}' and mimeType='application/vnd.google-apps.folder' "
    "and trashed=false and 'root' in parents"
)
found = drive_svc.files().list(q=q, fields='files(id, name)').execute().get('files', [])
if found:
    folder_id = found[0]['id']
    print(f'[1] 폴더 찾음: {FOLDER_NAME}/ (id={folder_id})')
else:
    folder = drive_svc.files().create(
        body={'name': FOLDER_NAME, 'mimeType': 'application/vnd.google-apps.folder'},
        fields='id',
    ).execute()
    folder_id = folder['id']
    print(f'[1] 폴더 생성: {FOLDER_NAME}/ (id={folder_id})')
    print(f'   ⚠️ 폴더가 막 생성됐어요. 첫 실행이면 Drive 웹에서:')
    print(f'      {FOLDER_NAME}/ 우클릭 → 공유 → "링크 있는 모든 사용자(뷰어)"로 설정')
    print(f'      그 뒤 이 셀을 다시 실행하세요.')

# 3) vocals.wav를 Drive 공개 폴더로 복사 — 마운트된 경로 통해 그냥 cp
drive_folder_path = Path('/content/drive/MyDrive') / FOLDER_NAME
drive_folder_path.mkdir(exist_ok=True)
drive_vocals = drive_folder_path / f'{STEM}_vocals.wav'
print(f'\n[2] Drive에 복사 중... ({vocals_path.stat().st_size / 1e6:.1f} MB)')
shutil.copy(vocals_path, drive_vocals)
print(f'   저장됨: {drive_vocals}')

# 4) 방금 복사한 파일의 Drive FILE_ID 조회 — 폴더에 같은 이름이 있을 수 있어 가장 최신을 택함
print('\n[3] FILE_ID 조회 중 (Drive 인덱싱 잠시 대기)...')
file_id = None
for attempt in range(20):
    q2 = (
        f"name='{drive_vocals.name}' and '{folder_id}' in parents and trashed=false"
    )
    res = drive_svc.files().list(
        q=q2,
        orderBy='modifiedTime desc',
        fields='files(id, name, modifiedTime)',
    ).execute().get('files', [])
    if res:
        file_id = res[0]['id']
        print(f'   FILE_ID: {file_id}')
        break
    time.sleep(2)
else:
    raise RuntimeError('Drive 인덱싱 대기 timeout — 잠시 후 셀 재실행하거나 Drive 웹에서 파일 확인')

# 5) 직접 다운로드 URL 구성
public_url = f'https://drive.google.com/uc?export=download&id={file_id}'
print(f'   공개 URL: {public_url}')

# 6) Audioshake task 생성
HEADERS = {'x-api-key': AUDIOSHAKE_API_KEY, 'Content-Type': 'application/json'}
BASE = 'https://api.audioshake.ai'

TARGETS = [
    {'model': 'leadVocal', 'formats': ['wav']},
    {'model': 'backingVocal', 'formats': ['wav']},
]

print(f'\n[4] task 생성 — targets: {[t["model"] for t in TARGETS]}')
r = requests.post(f'{BASE}/tasks',
                  headers=HEADERS,
                  json={'url': public_url, 'targets': TARGETS},
                  timeout=30)
print(f'   응답 코드: {r.status_code}')
print(f'   응답 본문: {r.text[:500]}')
r.raise_for_status()
task = r.json()
task_id = task.get('id') or task.get('taskId') or task.get('task_id')
print(f'   task_id: {task_id}')

# 7) 폴링 — top-level status가 없고 completedAt이 채워지면 끝.
#    target별 status도 함께 보면서 진행 상황 표시.
print(f'\n[5] 처리 대기...')
for attempt in range(120):
    time.sleep(5)
    s = requests.get(f'{BASE}/tasks/{task_id}',
                     headers={'x-api-key': AUDIOSHAKE_API_KEY},
                     timeout=20)
    s.raise_for_status()
    task = s.json()
    completed = task.get('completedAt')
    target_states = [(t.get('model'), t.get('status')) for t in task.get('targets', [])]
    state_str = ', '.join(f'{m}={s}' for m, s in target_states) or '대기 중'
    print(f'   [{attempt*5:3d}s] {state_str}')
    if completed:
        break
    if any(s in ('failed', 'error') for _, s in target_states):
        raise RuntimeError(f'task 실패: {task}')
else:
    raise TimeoutError('10분 안에 끝나지 않음')

# 8) 결과 다운로드 — 각 target의 output[0].link 사용
print(f'\n[6] 결과 다운로드:')
AS_DIR = WORK_DIR / 'audioshake'
AS_DIR.mkdir(exist_ok=True)

as_files = {}
for item in task.get('targets', []):
    model = item.get('model', 'unknown')
    outputs = item.get('output') or []
    if not outputs:
        print(f'   ⚠️ {model}: output 비어있음 — item: {item}')
        continue
    link = outputs[0].get('link') or outputs[0].get('url')
    if not link:
        print(f'   ⚠️ {model}: link 못 찾음 — outputs[0]={outputs[0]}')
        continue
    nice = 'lead' if 'lead' in model.lower() else ('backing' if 'back' in model.lower() else model)
    out = AS_DIR / f'{STEM}_audioshake_{nice}.wav'
    print(f'   - {model} → {out.name}')
    with requests.get(link, stream=True, timeout=120) as dr:
        dr.raise_for_status()
        with open(out, 'wb') as f:
            for chunk in dr.iter_content(8192):
                f.write(chunk)
    as_files[nice] = out

# 9) FINAL_DIR로 복사
for nice, p in as_files.items():
    shutil.copy(p, FINAL_DIR / p.name)
    print(f'✅ {p.name} ({p.stat().st_size / 1e6:.1f} MB)')

## 5. 재생 — 결과 비교

각 트랙을 직접 들어보면서 분리 품질 확인. **메인 vs 백보컬**이 핵심이고, 참고용으로 보컬 합본·반주도 같이 띄움.

In [ ]:
from IPython.display import Audio, display, HTML

def show_player(label: str, path: Path):
    display(HTML(f'<h4>{label} &mdash; <code>{path.name}</code></h4>'))
    display(Audio(str(path)))

# FINAL_DIR에 있는 결과를 의미 있는 순서로 표시.
# 파일명 패턴에 따라 라벨을 매기고, 존재하는 것만 띄움.
ORDER = [
    (f'{STEM}_audioshake_lead.wav',    '🎤 [Audioshake] 메인 보컬 (lead)'),
    (f'{STEM}_audioshake_backing.wav', '🎵 [Audioshake] 백보컬 / 화음 (backing)'),
    (f'{STEM}_lead.wav',               '🎤 [UVR] 메인 보컬 (lead)'),
    (f'{STEM}_backing.wav',            '🎵 [UVR] 백보컬 / 화음 (backing)'),
    (f'{STEM}_vocals.wav',             '🗣️ 보컬 전체 (Demucs vocals — 참고)'),
    (f'{STEM}_instrumental.wav',       '🎹 반주 (Demucs no_vocals — 참고)'),
]

shown = 0
for fname, label in ORDER:
    p = FINAL_DIR / fname
    if p.exists():
        show_player(label, p)
        shown += 1

# 위 패턴에 안 잡힌 wav가 더 있으면 마지막에 추가로 표시
known = {FINAL_DIR / n for n, _ in ORDER}
extras = [p for p in sorted(FINAL_DIR.glob('*.wav')) if p not in known]
for p in extras:
    show_player(f'📦 (기타) {p.name}', p)
    shown += 1

if shown == 0:
    print('⚠️ FINAL_DIR에 wav가 하나도 없습니다. §4.1~4.3 또는 §4.5를 먼저 실행하세요.')

## 6. 다운로드 — ZIP으로 묶어서

최종 4개 wav를 한 ZIP으로 묶어 다운로드.

In [ ]:
import zipfile
from google.colab import files

# FINAL_DIR 안의 모든 wav를 ZIP으로 묶음
zip_path = WORK_DIR / f'{STEM}_separation.zip'
wavs = sorted(FINAL_DIR.glob('*.wav'))
if not wavs:
    raise RuntimeError('FINAL_DIR에 wav가 없습니다 — 분리 단계를 먼저 실행하세요.')

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in wavs:
        zf.write(p, arcname=p.name)
        print(f'  + {p.name} ({p.stat().st_size / 1e6:.1f} MB)')

print(f'\nZIP 생성: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)')
files.download(str(zip_path))

## 7. (선택) 다른 모델 시도

Karaoke 분리는 곡마다 품질 편차가 있어요. 다른 모델로 비교해보고 싶을 때:

```python
# 옵션 1) 더 보수적인 모델 — backing 누설 적음
separator.load_model(model_filename="6_HP-Karaoke-UVR.pth")

# 옵션 2) MDX23 계열 — 더 최신, lead 깔끔
separator.load_model(model_filename="UVR_MDXNET_KARA_2.onnx")
```

사용 가능한 모델 전체 목록:
```python
from audio_separator.separator import Separator
Separator().list_supported_model_files()
```

## 알아둘 점

- **분리 한계**: 화음이 메인과 같은 음역대를 부르거나(unison) 메인에 강한 reverb·딜레이가 걸려 있으면 backing 트랙에 메인이 누설됨.
- **K-pop 특성**: 화음을 stereo wide하게 깔아두는 곡일수록 잘 분리됨. 모노 가깝게 믹싱된 곡은 어려움.
- **품질 비교**: lead 트랙에서 화음이 들리면 모델을 바꾸거나(§7), 신호처리적으로 backing을 lead에서 빼는 spectral subtraction을 추가로 적용 가능.